# Time-Based Usage Patterns and Student Profiles

This notebook derives temporal indicators of AI tutor usage from the cleaned chat-level database.

The main classification of students into occasional and regular users is based on the number of distinct days on which they used the tutor. Additional time-gap measures are retained as consistency checks and descriptive indicators of usage continuity.

The notebook also derives incentive-period indicators, exam-panic periods, and usage characteristics split by rewarded and non-rewarded periods.

The final user-level database is constructed in a later notebook.

## Load the cleaned chat-level database

The database contains one row for each question-answer pair. Student identifiers are pseudonymized.

In [1]:
import pandas as pd
df = pd.read_excel('databases/chat_database_clean.xlsx')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   chat_id              990 non-null    int64  
 1   question_id          990 non-null    int64  
 2   question             990 non-null    object 
 3   answer               990 non-null    object 
 4   q_time               990 non-null    object 
 5   a_time               990 non-null    object 
 6   id                   990 non-null    object 
 7   T1                   990 non-null    float64
 8   T2                   990 non-null    float64
 9   T3                   990 non-null    float64
 10  T4                   990 non-null    float64
 11  T1234                990 non-null    float64
 12  question_length      990 non-null    int64  
 13  answer_length        990 non-null    int64  
 14  question_count       990 non-null    int64  
 15  chat_count           990 non-null    int

### Creating the user variable 

Using the timestamp calculating active days and temporal gaps between questions

## Prepare timestamps

Question timestamps are converted to datetime format before deriving temporal indicators.

In [3]:
df['q_time'] = pd.to_datetime(df['q_time'])

In [4]:
print(df[["q_time", "a_time"]].dtypes)
print(df["q_time"].min())
print(df["q_time"].max())

q_time    datetime64[ns, UTC]
a_time                 object
dtype: object
2025-03-10 22:45:24.500275+00:00
2025-06-18 09:22:08.992833+00:00


## Active days

Students are classified primarily according to the number of distinct calendar days on which they submitted at least one question.

A threshold of seven active days is used to distinguish sustained usage from occasional usage.

In [5]:
df["q_date"] = df["q_time"].dt.date

active_days = (
    df.groupby("id")["q_date"]
      .nunique()
      .reset_index(name="active_days")
)

df = df.merge(active_days, on="id", how="left")

In [6]:
print(active_days["active_days"].describe())
print(
    active_days["active_days"]
    .value_counts()
    .sort_index()
)

count    44.000000
mean      5.545455
std       5.840733
min       1.000000
25%       1.000000
50%       3.000000
75%       8.250000
max      22.000000
Name: active_days, dtype: float64
active_days
1     14
2      6
3      5
4      2
5      2
6      2
7      1
8      1
9      2
10     1
11     1
14     2
15     1
16     1
17     1
21     1
22     1
Name: count, dtype: int64


## User profile

Students with at least seven active days are classified as regular users. All other registered users are classified as occasional users.

This criterion captures sustained engagement more directly than elapsed time alone.

In [7]:
import numpy as np
df["profile"] = np.where(
    df["active_days"] >= 7,
    "regular",
    "occasional"
)

In [8]:
profile_check = (
    df[["id", "profile"]]
    .drop_duplicates()
    ["profile"]
    .value_counts()
    .sort_index()
)

print(profile_check)

profile
occasional    31
regular       13
Name: count, dtype: int64


## Time-gap diagnostics

To assess the temporal consistency of the active-day classification, we calculate the elapsed time between the first and last question, the maximum gap between consecutive questions, and the average gap.

These indicators are used as diagnostic measures rather than as the primary classification criterion.

In [9]:
def calculate_time_gaps(df):
    results = []

    for student_id, group in df.groupby('id'):
        times = group['q_time'].sort_values()

        total_duration = (times.max() - times.min()).total_seconds()

        # Time gaps between subsequent questions
        time_gaps = times.diff().dt.total_seconds().dropna()

        # Biggest and average time gaps 
        max_gap = time_gaps.max() if not time_gaps.empty else 0
        avg_gap = time_gaps.mean() if not time_gaps.empty else 0

        results.append({
            'id': student_id,
            'total_duration': total_duration,
            'max_gap': max_gap,
            'avg_gap': avg_gap
        })

    return pd.DataFrame(results)


df_times = calculate_time_gaps(df)

In [10]:
df_times = df_times.merge(
    df[['id', 'profile']].drop_duplicates(),
    on='id',
    how='left'
)

In [11]:
regular_check = df_times[df_times['profile'] == 'regular'].copy()

regular_check['duration_ok'] = regular_check['total_duration'] >= 2e6
regular_check['max_gap_ok'] = regular_check['max_gap'] <= 4e6
regular_check['avg_gap_ok'] = regular_check['avg_gap'] <= 2e5

regular_check

,id,total_duration,max_gap,avg_gap,profile,duration_ok,max_gap_ok,avg_gap_ok
11,U132,5.617865e+06,1.214105e+06,100319.026360,regular,True,True,True
12,U133,6.844260e+06,1.411501e+06,82460.958652,regular,True,True,True
16,U20,2.358183e+06,6.240507e+05,76070.408180,regular,True,True,True
21,U39,4.924013e+06,1.185698e+06,58619.206826,regular,True,True,True
25,U5,5.698516e+06,1.206027e+06,73057.898088,regular,True,True,True
26,U52,3.008760e+06,1.380027e+06,150437.979565,regular,True,True,True
30,U64,5.939707e+06,1.390815e+06,179991.125346,regular,True,True,True
31,U65,5.436520e+06,1.207942e+06,95377.536449,regular,True,True,True
32,U73,6.060733e+06,1.611157e+06,159492.977162,regular,True,True,True
37,U81,5.861826e+06,7.645423e+05,53289.330028,regular,True,True,True


In [12]:
regular_check[
    ['duration_ok', 'max_gap_ok', 'avg_gap_ok']
].all()

duration_ok    True
max_gap_ok     True
avg_gap_ok     True
dtype: bool

In [13]:
regular_check[
    ['duration_ok', 'max_gap_ok', 'avg_gap_ok']
].sum()

duration_ok    13
max_gap_ok     13
avg_gap_ok     13
dtype: int64

### Gap-based consistency checks

The published study describes regular users as having:

- at least 7 active days;
- at least 2 million seconds between the first and last questions (~23 days);
- no gap exceeding 4 million seconds between consecutive questions (~46 days);
- an average gap of no more than two hundred thousand seconds.

In the present workflow, the seven-day criterion defines the profile, while the remaining conditions are retained as diagnostic checks.

## Course relevance

The average relevance rate is calculated for each student.

In [14]:
rel = (
    df.groupby("id")["relevance"]
      .mean()
      .reset_index(name="relevance_rate")
)

df = df.merge(rel, on="id", how="left")

## Rewarded and non-rewarded periods

The incentive period ended on May 5, 2025 at 16:00.

Questions submitted after this point are classified as non-rewarded.

In [18]:
cutoff = pd.Timestamp("2025-05-05 16:00:00")
df['q_time'] = df['q_time'].dt.tz_localize(None)
df["non_rewarded"] = (
    df["q_time"] > cutoff
).astype(int)

In [19]:
period_check = df["non_rewarded"].value_counts().sort_index()

print(period_check)

print("\nShare:")
print(df["non_rewarded"].mean())

non_rewarded
0    866
1    124
Name: count, dtype: int64

Share:
0.12525252525252525


In [20]:
period = (
    df.groupby("id")["non_rewarded"]
      .agg(
          non_rewarded_count="sum",
          total_questions="count"
      )
      .reset_index()
)

period["non_rewarded_share"] = (
    period["non_rewarded_count"] /
    period["total_questions"]
)

period["rewarded_count"] = (
    period["total_questions"] -
    period["non_rewarded_count"]
)

period["rewarded_share"] = (
    period["rewarded_count"] /
    period["total_questions"]
)

df = df.merge(period, on="id", how="left")

## Active days by incentive period

The number of distinct days with AI activity is calculated separately for the rewarded and non-rewarded periods.

In [21]:
days_rew = (
    df.loc[df["non_rewarded"] == 0]
      .groupby("id")["q_date"]
      .nunique()
      .reset_index(name="active_days_rew")
)

days_post = (
    df.loc[df["non_rewarded"] == 1]
      .groupby("id")["q_date"]
      .nunique()
      .reset_index(name="active_days_post")
)

df = df.merge(days_rew, on="id", how="left")
df = df.merge(days_post, on="id", how="left")

df["active_days_post"] = df["active_days_post"].fillna(0)

## Exam-panic periods

We flag questions submitted during the two to three days preceding the second, third, and fourth tests.

The resulting indicator is used descriptively to examine whether AI usage increased before examinations.

In [22]:
panic_periods = [
    ("2025-03-29", "2025-03-31 16:00:00"),
    ("2025-05-03", "2025-05-05 16:00:00"),
    ("2025-05-17", "2025-05-20 16:00:00")
]

panic = pd.Series(False, index=df.index)

for start, end in panic_periods:
    panic |= (
        (df["q_time"] > pd.Timestamp(start)) &
        (df["q_time"] < pd.Timestamp(end))
    )

df["exam_panic"] = panic.astype(int)

In [23]:
panic_rate = (
    df.groupby("id")["exam_panic"]
      .mean()
      .reset_index(name="exam_panic_rate")
)

df = df.merge(panic_rate, on="id", how="left")

In [24]:
print(df["exam_panic"].value_counts().sort_index())
print(
    df.loc[df["exam_panic"] == 1, "id"]
      .nunique(),
    "students contributed to exam-panic queries."
)

exam_panic
0    834
1    156
Name: count, dtype: int64
17 students contributed to exam-panic queries.


## Question and answer length by incentive period

Question and answer lengths are summarized separately for the rewarded and non-rewarded periods.

In [25]:
df["q_len_rew"] = df.loc[
    df["non_rewarded"] == 0, "question_length"
]

df["q_len_post"] = df.loc[
    df["non_rewarded"] == 1, "question_length"
]

df["a_len_rew"] = df.loc[
    df["non_rewarded"] == 0, "answer_length"
]

df["a_len_post"] = df.loc[
    df["non_rewarded"] == 1, "answer_length"
]

In [26]:
lengths = (
    df.groupby("id")[
        ["q_len_rew", "q_len_post", "a_len_rew", "a_len_post"]
    ]
    .mean()
    .reset_index()
    .rename(columns={
        "q_len_rew": "avg_q_len_rew",
        "q_len_post": "avg_q_len_post",
        "a_len_rew": "avg_a_len_rew",
        "a_len_post": "avg_a_len_post"
    })
)

df = df.merge(lengths, on="id", how="left")

## Course relevance by incentive period

The proportion of relevant questions is summarized separately for the rewarded and non-rewarded periods.

In [27]:
df["rel_rew"] = df.loc[
    df["non_rewarded"] == 0, "relevance"
]

df["rel_post"] = df.loc[
    df["non_rewarded"] == 1, "relevance"
]

rel_period = (
    df.groupby("id")[
        ["rel_rew", "rel_post"]
    ]
    .mean()
    .reset_index()
    .rename(columns={
        "rel_rew": "avg_rel_rew",
        "rel_post": "avg_rel_post"
    })
)

df = df.merge(rel_period, on="id", how="left")

## Final data checks

The following checks document the main sample size, user count, profile composition, and incentive-period distribution.

In [28]:
print("Rows:", len(df))
print("Registered users:", df["id"].nunique())

print("\nProfile:")
print(
    df[["id", "profile"]]
    .drop_duplicates()["profile"]
    .value_counts()
)

print("\nReward period:")
print(df["non_rewarded"].value_counts().sort_index())

print("\nActive days:")
print(df[["id", "active_days"]].drop_duplicates()["active_days"].describe())

Rows: 990
Registered users: 44

Profile:
profile
occasional    31
regular       13
Name: count, dtype: int64

Reward period:
non_rewarded
0    866
1    124
Name: count, dtype: int64

Active days:
count    44.000000
mean      5.545455
std       5.840733
min       1.000000
25%       1.000000
50%       3.000000
75%       8.250000
max      22.000000
Name: active_days, dtype: float64


In [29]:
profile_users = (
    df[["id", "profile"]]
    .drop_duplicates()
)

profile_queries = (
    df.groupby("profile")
      .size()
      .sort_index()
)

print("Users by profile:")
print(profile_users["profile"].value_counts().sort_index())

print("\nQueries by profile:")
print(profile_queries)

Users by profile:
profile
occasional    31
regular       13
Name: count, dtype: int64

Queries by profile:
profile
occasional    289
regular       701
dtype: int64


## Save the temporal analysis dataset

The resulting dataframe remains at the chat level. User-level aggregation and the final analytical database are constructed in a later notebook.

In [30]:
df.to_excel(
    "databases/chat_database_time.xlsx",
    index=False
)